<a href="https://colab.research.google.com/github/rapaaal/Project-with-Google-Collab/blob/main/online_store_ordering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# mendefinisikan item produk yang di jual
class Produk:
    def __init__(self, id_produk, nama, harga, stok):
        self.id_produk = id_produk
        self.nama = nama
        self.harga = harga
        self.stok = stok

    def cek_ketersediaan(self, jumlah):
        # Mengembalikan True jika stok cukup, False jika tidak
        return self.stok >= jumlah

    def kurangi_stok(self, jumlah):
        # Mengurangi stok setelah pesanan diproses
        if self.cek_ketersediaan(jumlah):
            self.stok -= jumlah
            return True
        return False


In [20]:
# menampung produk yang ingin dibeli
class KeranjangBelanja:
    def __init__(self):
        # Menggunakan dictionary: {objek_produk: jumlah_pesanan}
        self.items = {}

    def tambah_item(self, produk, jumlah):
        # Logika validasi stok sebelum masuk keranjang
        if produk.cek_ketersediaan(jumlah):
            if produk in self.items:
                self.items[produk] += jumlah
            else:
                self.items[produk] = jumlah
            print(f"✅ Berhasil: {jumlah} {produk.nama} ditambahkan ke keranjang.")
        else:
            print(f"❌ Gagal: Stok {produk.nama} tidak mencukupi! (Sisa stok: {produk.stok})")

    def hitung_subtotal(self):
        # Menghitung total harga semua barang di keranjang
        total = 0
        for produk, jumlah in self.items.items():
            total += produk.harga * jumlah
        return total

    def tampilkan_isi(self):
        print("\n🛒 --- ISI KERANJANG ANDA ---")
        if not self.items:
            print("Keranjang masih kosong.")
            return

        for produk, jumlah in self.items.items():
            subtotal_item = produk.harga * jumlah
            print(f"- {produk.nama:<15} (x{jumlah}) : Rp {subtotal_item:,.0f}")
        print("-" * 30)
        print(f"Subtotal Sementara: Rp {self.hitung_subtotal():,.0f}\n")



In [25]:
# memproses checkout dan logika pembayaran/diskon
class Pesanan:
    def __init__(self, nama_pelanggan, keranjang):
        self.pelanggan = nama_pelanggan
        self.keranjang = keranjang
        self.status = "Menunggu Pembayaran"
        self.total_akhir = 0

    def hitung_diskon(self, subtotal):
        # LOGIKA DISKON: Diskon 10% jika belanja di atas Rp 500.000
        batas_diskon = 500000
        persentase_diskon = 0.10

        if subtotal > batas_diskon:
            return subtotal * persentase_diskon
        return 0
    def pilih_metode_pembayaran(self):
        print("\n💳 --- PILIH METODE PEMBAYARAN ---")
        print("1. Transfer Bank (Admin: Rp 0)")
        print("2. E-Wallet OVO/GoPay (Admin: Rp 1,000)")
        print("3. Kartu Kredit (Biaya Layanan 2%)")

        pilihan = input("Pilih metode (1-3): ")
        subtotal = self.keranjang.hitung_subtotal()

        if pilihan == '1':
            self.metode_bayar = "Transfer Bank"
            self.biaya_admin = 0
        elif pilihan == '2':
            self.metode_bayar = "E-Wallet"
            self.biaya_admin = 1000
        elif pilihan == '3':
            self.metode_bayar = "Kartu Kredit"
            self.biaya_admin = subtotal * 0.02
        else:
            print("⚠️ Pilihan tidak valid, otomatis memilih Transfer Bank.")
            self.metode_bayar = "Transfer Bank"
            self.biaya_admin = 0


    def proses_checkout(self):
        print(f"📦 Memproses pesanan atas nama: {self.pelanggan}...")

        if not self.keranjang.items:
            print("❌ Pesanan gagal: Keranjang kosong!")
            return

        subtotal = self.keranjang.hitung_subtotal()
        diskon = self.hitung_diskon(subtotal)
        self.total_akhir = subtotal - diskon

        # LOGIKA UPDATE STOK: Kurangi stok asli di gudang/database
        for produk, jumlah in self.keranjang.items.items():
            produk.kurangi_stok(jumlah)

        self.status = "Lunas & Diproses"
        self.cetak_struk(subtotal, diskon)

    def cetak_struk(self, subtotal, diskon):
        print("\n======================================")
        print("          STRUK PEMBELIAN             ")
        print("======================================")
        print(f"Pelanggan : {self.pelanggan}")
        print(f"Status    : {self.status}")
        print("--------------------------------------")
        for produk, jumlah in self.keranjang.items.items():
            total_harga_item = produk.harga * jumlah
            print(f"{produk.nama:<15} x{jumlah:<3} Rp {total_harga_item:>10,.0f}")
        print("--------------------------------------")
        print(f"Subtotal      : Rp {subtotal:>15,.0f}")
        print(f"Diskon        : Rp {diskon:>15,.0f}")
        print(f"TOTAL BAYAR   : Rp {self.total_akhir:>15,.0f}")
        print("======================================\n")


In [24]:
# Eksekusi Program
if __name__ == "__main__":
    print("--- SELAMAT DATANG DI TOKO ONLINE ---\n")

    # 1. Menyiapkan Data Produk di Gudang
    daftar_produk = {
        "1": Produk("P01", "Laptop Asus", 7500000, 5),
        "2": Produk("P02", "Mouse Logitec", 150000, 20),
        "3": Produk("P03", "Keyboard Mech", 450000, 2),
        "4": Produk("P04", "Flashdisk 64GB", 85000, 10)
    }

     # 2. Meminta input nama pelanggan
    nama_pelanggan = input("Masukkan nama Anda: ")
    keranjang = KeranjangBelanja()

    # 3. Loop Menu Interaktif
    while True:
        print("\n=== MENU UTAMA ===")
        print("1. Lihat Katalog Produk")
        print("2. Tambah Barang ke Keranjang")
        print("3. Lihat Isi Keranjang")
        print("4. Checkout (Bayar)")
        print("5. Keluar")

        pilihan = input("Pilih menu (1-5): ")

        if pilihan == '1':
            print("\n--- KATALOG PRODUK ---")
            for key, p in daftar_produk.items():
                print(f"[{key}] {p.nama:<15} - Rp {p.harga:>10,.0f} | Stok: {p.stok}")

        elif pilihan == '2':
            print("\n--- KATALOG PRODUK ---")
            for key, p in daftar_produk.items():
                print(f"[{key}] {p.nama:<15} - Rp {p.harga:>10,.0f} | Stok: {p.stok}")

            id_pilih = input("\nMasukkan nomor produk yang ingin dibeli (1-4): ")
            if id_pilih in daftar_produk:
                try:
                    jumlah = int(input(f"Berapa banyak {daftar_produk[id_pilih].nama} yang ingin dibeli? "))
                    if jumlah > 0:
                        keranjang.tambah_item(daftar_produk[id_pilih], jumlah)
                    else:
                        print("❌ Gagal: Jumlah barang harus lebih dari 0.")
                except ValueError:
                    print("❌ Gagal: Masukkan angka yang valid!")
            else:
                print("❌ Gagal: Nomor produk tidak ditemukan.")

        elif pilihan == '3':
            keranjang.tampilkan_isi()

        elif pilihan == '4':
            pesanan = Pesanan(nama_pelanggan, keranjang)
            pesanan.proses_checkout()

            # Cek apakah pesanan berhasil checkout (keranjang tidak kosong)
            if pesanan.status == "Lunas & Diproses":
                print("Terima kasih telah berbelanja! Pesanan Anda sedang diproses.")
                break # Keluar dari program setelah selesai belanja

        elif pilihan == '5':
            print(f"\nTerima kasih {nama_pelanggan} telah berkunjung. Sampai jumpa!")
            break

        else:
            print("❌ Pilihan tidak valid. Silakan masukkan angka 1-5.")

--- SELAMAT DATANG DI TOKO ONLINE ---

Masukkan nama Anda: rafa albanin

=== MENU UTAMA ===
1. Lihat Katalog Produk
2. Tambah Barang ke Keranjang
3. Lihat Isi Keranjang
4. Checkout (Bayar)
5. Keluar
Pilih menu (1-5): 1

--- KATALOG PRODUK ---
[1] Laptop Asus     - Rp  7,500,000 | Stok: 5
[2] Mouse Logitec   - Rp    150,000 | Stok: 20
[3] Keyboard Mech   - Rp    450,000 | Stok: 2
[4] Flashdisk 64GB  - Rp     85,000 | Stok: 10

=== MENU UTAMA ===
1. Lihat Katalog Produk
2. Tambah Barang ke Keranjang
3. Lihat Isi Keranjang
4. Checkout (Bayar)
5. Keluar
Pilih menu (1-5): 2

--- KATALOG PRODUK ---
[1] Laptop Asus     - Rp  7,500,000 | Stok: 5
[2] Mouse Logitec   - Rp    150,000 | Stok: 20
[3] Keyboard Mech   - Rp    450,000 | Stok: 2
[4] Flashdisk 64GB  - Rp     85,000 | Stok: 10

Masukkan nomor produk yang ingin dibeli (1-4): 1
Berapa banyak Laptop Asus yang ingin dibeli? 2
✅ Berhasil: 2 Laptop Asus ditambahkan ke keranjang.

=== MENU UTAMA ===
1. Lihat Katalog Produk
2. Tambah Barang ke K